<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Data Carpentry · Python profesional</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Pandas: estructuras, tipos, memoria y selección eficiente**

Entramos en el bloque de **Pandas profesional**. El objetivo no es repasar `DataFrame` y `Series` desde cero, sino entender qué decisiones aparentemente pequeñas pueden afectar a:

- corrección;
- legibilidad;
- memoria;
- velocidad;
- reproducibilidad;
- mantenibilidad.

Seguiremos el patrón del curso:

> **observar → medir → detectar → corregir → volver a medir → interpretar**

Trabajaremos con un dataset suficientemente grande para que algunas diferencias sean visibles.

## Objetivos

Al finalizar deberías poder:

1. Distinguir `Series` y `DataFrame`.
2. Inspeccionar estructura, índices y dtypes.
3. Detectar tipos inadecuados.
4. Medir memoria global y por columna.
5. Optimizar columnas categóricas.
6. Entender el coste de `object`.
7. Seleccionar correctamente con `[]`, `.loc` e `.iloc`.
8. Diferenciar selección por etiqueta y posición.
9. Detectar chained indexing.
10. Evitar asignaciones ambiguas.
11. Comprender copias y objetos derivados.
12. Construir filtros booleanos legibles.
13. Comparar estrategias mediante benchmarks.
14. Validar que una optimización conserva el resultado.

## Cómo trabajar

**Alumnado:** ejecuta la parte guiada y completa los doce ejercicios en orden. Sustituye los TODO por tus cálculos y comprobaciones. Las plantillas pueden ejecutarse sin completar la práctica, pero eso no demuestra que los ejercicios estén resueltos.

**Entorno:** Python, NumPy, Pandas y Matplotlib. Datos sintéticos autocontenidos de 200.000 filas. Los tiempos y la memoria dependen del entorno; las advertencias de asignación dependen de la versión y el modo de copia de Pandas.

La auditoría y los ejercicios de categoría se centran en object según el enunciado. Si tu entorno utiliza string, revisa esa representación también. El docente seleccionará los ejercicios de aula y las ampliaciones.


## 1. Preparación

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print("Pandas:", pd.__version__)

## 2. Crear un dataset de trabajo

In [ ]:
N = 200_000

ventas = pd.DataFrame({
    "venta_id": np.arange(1, N + 1),
    "cliente_id": rng.integers(1, 30_001, N),
    "canal": rng.choice(
        ["web", "tienda", "partner", "marketplace"],
        N,
        p=[0.45, 0.30, 0.15, 0.10]
    ),
    "region": rng.choice(
        ["Norte", "Sur", "Este", "Oeste", "Centro"],
        N
    ),
    "unidades": rng.integers(1, 10, N),
    "precio": rng.uniform(5, 250, N).round(2),
    "descuento": rng.choice(
        [0.0, 0.05, 0.10, 0.20],
        N,
        p=[0.55, 0.20, 0.18, 0.07]
    ),
    "fecha": pd.date_range(
        "2025-01-01",
        periods=N,
        freq="min"
    )
})

ventas["importe"] = (
    ventas["unidades"]
    * ventas["precio"]
    * (1 - ventas["descuento"])
)

ventas.head()

### Interpretación

Usaremos 200.000 observaciones. No es un dataset masivo, pero sí suficientemente grande para estudiar memoria y rendimiento sin depender de ficheros externos.

## 3. Series frente a DataFrame

In [ ]:
columna = ventas["precio"]
tabla = ventas[["precio"]]

print(type(columna))
print("Shape Series:", columna.shape)

print()
print(type(tabla))
print("Shape DataFrame:", tabla.shape)

Aunque contienen los mismos valores, no son el mismo objeto:

- `ventas["precio"]` produce una `Series`;
- `ventas[["precio"]]` mantiene estructura bidimensional y produce un `DataFrame`.

Esta diferencia importa al encadenar operaciones o diseñar funciones.

## 4. Inspección estructural antes de transformar

In [ ]:
print("Shape:", ventas.shape)
print("Índice:", type(ventas.index).__name__)
print()
print(ventas.dtypes)

### 4.1 `info()`

In [ ]:
ventas.info(memory_usage="deep")

`info()` ofrece una primera auditoría: número de filas, columnas, no nulos, tipos y memoria aproximada.

## 5. Memoria por columna

In [ ]:
memoria_columnas = (
    ventas.memory_usage(deep=True)
    .sort_values(ascending=False)
    .rename("bytes")
    .to_frame()
)

memoria_columnas["MB"] = memoria_columnas["bytes"] / 1024**2
memoria_columnas

### 5.1 Visualizar qué columnas consumen más

In [ ]:
mem_plot = memoria_columnas.drop(index="Index", errors="ignore")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(mem_plot.index, mem_plot["MB"])
ax.set(
    title="Memoria profunda por columna",
    xlabel="MB",
    ylabel="Columna"
)
plt.show()

### Interpretación

Las columnas de texto repetitivo suelen consumir bastante memoria cuando permanecen como `object`. No optimizaremos todavía: primero hemos cuantificado el problema.

## 6. Detectar cardinalidad

In [ ]:
cardinalidad = pd.DataFrame({
    "dtype": ventas.dtypes.astype(str),
    "n_unicos": ventas.nunique(dropna=False),
    "pct_unicos": ventas.nunique(dropna=False) / len(ventas) * 100
})

cardinalidad

Una columna textual con muy pocos valores distintos respecto al número de filas es candidata a representación categórica.

## 7. Optimizar categorías

In [ ]:
ventas_opt = ventas.copy()

for columna in ["canal", "region"]:
    ventas_opt[columna] = ventas_opt[columna].astype("category")

print(ventas_opt.dtypes)

### 7.1 Medir antes y después

In [ ]:
mem_antes = ventas.memory_usage(deep=True).sum() / 1024**2
mem_despues = ventas_opt.memory_usage(deep=True).sum() / 1024**2

comparacion_memoria = pd.DataFrame({
    "version": ["object", "category"],
    "memoria_MB": [mem_antes, mem_despues]
})

comparacion_memoria["reduccion_pct"] = (
    1 - comparacion_memoria["memoria_MB"] / mem_antes
) * 100

comparacion_memoria

### 7.2 Visualizar la reducción

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    comparacion_memoria["version"],
    comparacion_memoria["memoria_MB"]
)
ax.set(
    title="Memoria antes y después de categorizar",
    xlabel="Representación",
    ylabel="Memoria total (MB)"
)
plt.show()

### Interpretación

La optimización no cambia el significado de `canal` o `region`; cambia su representación interna. En columnas de baja cardinalidad puede reducir notablemente la memoria.

## 8. Validar que la optimización conserva valores

In [ ]:
for columna in ["canal", "region"]:
    iguales = (
        ventas[columna].astype(str).to_numpy()
        == ventas_opt[columna].astype(str).to_numpy()
    ).all()

    print(columna, "->", iguales)

Nunca deberíamos considerar válida una optimización únicamente porque consume menos. También debemos comprobar que conserva la información esperada.

## 9. Tipos numéricos y rango real

In [ ]:
ventas[["venta_id", "cliente_id", "unidades", "precio", "descuento"]].dtypes

### 9.1 Inspeccionar rangos

In [ ]:
ventas[[
    "venta_id",
    "cliente_id",
    "unidades",
    "precio",
    "descuento"
]].agg(["min", "max"])

Los rangos ayudan a decidir si un dtype es sobredimensionado. Optimizar enteros o floats requiere más cuidado que convertir categorías, porque puede afectar precisión o rango.

## 10. Downcasting controlado

In [ ]:
ventas_down = ventas_opt.copy()

ventas_down["venta_id"] = pd.to_numeric(
    ventas_down["venta_id"],
    downcast="unsigned"
)

ventas_down["cliente_id"] = pd.to_numeric(
    ventas_down["cliente_id"],
    downcast="unsigned"
)

ventas_down["unidades"] = pd.to_numeric(
    ventas_down["unidades"],
    downcast="unsigned"
)

ventas_down[["venta_id", "cliente_id", "unidades"]].dtypes

### 10.1 Volver a medir

In [ ]:
versiones_memoria = pd.DataFrame({
    "version": [
        "Original",
        "Categorías",
        "Categorías + downcast"
    ],
    "MB": [
        ventas.memory_usage(deep=True).sum() / 1024**2,
        ventas_opt.memory_usage(deep=True).sum() / 1024**2,
        ventas_down.memory_usage(deep=True).sum() / 1024**2
    ]
})

versiones_memoria["ahorro_vs_original_pct"] = (
    1 - versiones_memoria["MB"] / versiones_memoria.loc[0, "MB"]
) * 100

versiones_memoria

### 10.2 Gráfico acumulado de optimización

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(versiones_memoria["version"], versiones_memoria["MB"])
ax.set(
    title="Impacto acumulado de la optimización de dtypes",
    ylabel="Memoria (MB)"
)
ax.tick_params(axis="x", rotation=20)
plt.show()

### Interpretación

Optimizar dtypes debe ser una decisión basada en:

1. rango;
2. precisión necesaria;
3. memoria;
4. compatibilidad con operaciones posteriores.

No debe hacerse de forma ciega.

# 11. Selección con `[]`

In [ ]:
ventas["precio"].head()

In [ ]:
ventas[["precio", "unidades", "importe"]].head()

`[]` es cómodo para seleccionar columnas, pero cuando queremos combinar filas y columnas de forma explícita conviene utilizar `.loc` o `.iloc`.

# 12. `.loc`: selección por etiqueta

In [ ]:
ventas.loc[
    0:4,
    ["venta_id", "canal", "importe"]
]

Con `.loc`, tanto filas como columnas se expresan mediante **etiquetas**.

Atención: en un `RangeIndex`, `0:4` incluye la etiqueta `4`.

# 13. `.iloc`: selección por posición

In [ ]:
ventas.iloc[
    0:5,
    [0, 2, 7]
]

`.iloc` trabaja con posiciones enteras y sigue la semántica habitual de slicing de Python: el extremo final no se incluye.

# 14. Etiqueta no es posición

In [ ]:
ejemplo_indice = pd.DataFrame(
    {"valor": [10, 20, 30, 40]},
    index=[100, 200, 300, 400]
)

ejemplo_indice

In [ ]:
print("loc[200]:")
display(ejemplo_indice.loc[200])

print("iloc[1]:")
display(ejemplo_indice.iloc[1])

Ambas expresiones llegan a la misma fila en este caso, pero por mecanismos diferentes:

- `.loc[200]`: etiqueta 200;
- `.iloc[1]`: segunda posición.

# 15. Provocar un error de selección

In [ ]:
try:
    ejemplo_indice.loc[1]
except KeyError as error:
    print(type(error).__name__, "->", error)

La etiqueta `1` no existe, aunque sí exista una segunda posición. Confundir posición y etiqueta puede producir errores o, peor, seleccionar datos incorrectos.

# 16. Filtros booleanos: empezar con uno simple

In [ ]:
ventas_web = ventas.loc[
    ventas["canal"] == "web"
]

ventas_web.head()

## 16.1 Varias condiciones

In [ ]:
ventas_interes = ventas.loc[
    (ventas["canal"] == "web")
    & (ventas["importe"] >= 1000)
    & (ventas["descuento"] <= 0.10),
    ["venta_id", "canal", "importe", "descuento"]
]

ventas_interes.head()

### Importante

Cada condición se encierra entre paréntesis y se combinan con:

- `&` para AND;
- `|` para OR;
- `~` para NOT.

No se utilizan `and` y `or` sobre Series.

# 17. Mostrar el error con `and`

In [ ]:
try:
    mascara_mal = (
        (ventas["canal"] == "web")
        and
        (ventas["importe"] > 1000)
    )
except ValueError as error:
    print(type(error).__name__, "->", error)

## 17.1 Corrección

In [ ]:
mascara_correcta = (
    (ventas["canal"] == "web")
    &
    (ventas["importe"] > 1000)
)

print("Filas seleccionadas:", mascara_correcta.sum())

# 18. Máscaras con nombre para mejorar legibilidad

In [ ]:
es_web = ventas["canal"].eq("web")
es_venta_alta = ventas["importe"].ge(1000)
descuento_bajo = ventas["descuento"].le(0.10)

seleccion = ventas.loc[
    es_web & es_venta_alta & descuento_bajo
]

seleccion.head()

Nombrar máscaras puede ser preferible cuando una regla de selección tiene significado de negocio. Facilita lectura, depuración y reutilización.

# 19. `isin()` frente a cadenas de OR

In [ ]:
regiones_objetivo = ["Norte", "Este", "Centro"]

filtro_isin = ventas["region"].isin(regiones_objetivo)

ventas.loc[
    filtro_isin,
    ["venta_id", "region", "importe"]
].head()

`isin()` suele expresar con más claridad la pertenencia a un conjunto que una larga secuencia de comparaciones con `|`.

# 20. `between()`

In [ ]:
ventas.loc[
    ventas["importe"].between(500, 1000),
    ["venta_id", "importe"]
].head()

# 21. Seleccionar solo lo necesario

In [ ]:
columnas_analisis = [
    "venta_id",
    "canal",
    "region",
    "importe"
]

ventas_reducidas = ventas.loc[
    :,
    columnas_analisis
]

ventas_reducidas.head()

### 21.1 Medir memoria

In [ ]:
mem_comparacion = pd.DataFrame({
    "objeto": ["DataFrame completo", "Columnas necesarias"],
    "MB": [
        ventas.memory_usage(deep=True).sum() / 1024**2,
        ventas_reducidas.memory_usage(deep=True).sum() / 1024**2
    ]
})

mem_comparacion

### 21.2 Visualización

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(mem_comparacion["objeto"], mem_comparacion["MB"])
ax.set(
    title="Memoria según columnas conservadas",
    ylabel="MB"
)
ax.tick_params(axis="x", rotation=15)
plt.show()

### Interpretación

Reducir columnas temprano puede disminuir la memoria de los objetos intermedios. En pipelines grandes, seleccionar solo lo necesario también mejora claridad.

# 22. Chained indexing: primero la mala práctica

In [ ]:
ejemplo_chained = ventas.head(20).copy()

subset = ejemplo_chained[
    ejemplo_chained["canal"] == "web"
]

subset["segmento_importe"] = "web"

subset.head()

### ¿Por qué es problemático?

Expresiones del estilo:

```python
df[condicion]["columna"] = valor
```

mezclan selección y asignación en pasos cuya semántica puede ser ambigua. Dependiendo del objeto obtenido, podemos estar trabajando sobre una copia o un objeto derivado.

La alternativa explícita es `.loc`.

## 22.1 Asignación explícita con `.loc`

In [ ]:
ejemplo_loc = ventas.head(20).copy()

mascara = ejemplo_loc["canal"].eq("web")

ejemplo_loc.loc[
    mascara,
    "segmento_importe"
] = "web"

ejemplo_loc.head()

### Interpretación

`.loc[filas, columna] = valor` expresa en una única operación qué filas y qué columna queremos modificar.

# 23. Comprobar el objeto original

In [ ]:
original = ventas.head(10).copy()
derivado = original.loc[
    original["canal"] == "web"
].copy()

derivado["precio"] = 0

print("DERIVADO")
display(derivado[["venta_id", "precio"]])

print("ORIGINAL")
display(original[["venta_id", "precio"]])

El `.copy()` explícito comunica que queremos un objeto independiente antes de modificarlo. Esto evita depender de detalles internos sobre vistas y copias.

# 24. `.query()` como alternativa expresiva

In [ ]:
resultado_query = ventas.query(
    "canal == 'web' and importe >= 1000 and descuento <= 0.10"
)

resultado_query.head()

`.query()` puede ser muy legible en expresiones sencillas. No es automáticamente mejor ni más rápido: debe elegirse por claridad y medirse cuando el rendimiento sea relevante.

# 25. Benchmark: máscara booleana vs query

In [ ]:
def medir(funcion, repeticiones=7):
    tiempos = []

    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion()
        tiempos.append(time.perf_counter() - inicio)

    return np.array(tiempos)


def filtro_booleano():
    return ventas.loc[
        (ventas["canal"] == "web")
        & (ventas["importe"] >= 1000)
        & (ventas["descuento"] <= 0.10)
    ]


def filtro_query():
    return ventas.query(
        "canal == 'web' and importe >= 1000 and descuento <= 0.10"
    )


t_bool = medir(filtro_booleano)
t_query = medir(filtro_query)

benchmark = pd.DataFrame({
    "metodo": ["Máscara booleana", "query"],
    "media_ms": [
        t_bool.mean() * 1000,
        t_query.mean() * 1000
    ],
    "std_ms": [
        t_bool.std() * 1000,
        t_query.std() * 1000
    ]
})

benchmark

## 25.1 Visualizar benchmark

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    benchmark["metodo"],
    benchmark["media_ms"],
    yerr=benchmark["std_ms"],
    capsize=4
)

ax.set(
    title="Tiempo medio de filtrado",
    ylabel="Milisegundos"
)

plt.show()

### Interpretación

El resultado depende del tamaño, expresión, versión de Pandas y hardware. La conclusión metodológica es más importante que quién gane una ejecución concreta:

> **medir antes de afirmar que una alternativa es más rápida.**

# 26. Validar equivalencia de ambos filtros

In [ ]:
a = filtro_booleano().sort_index()
b = filtro_query().sort_index()

print("Mismo shape:", a.shape == b.shape)
print("Mismos índices:", a.index.equals(b.index))

Una comparación de rendimiento solo tiene sentido si las alternativas producen el mismo resultado lógico.

# 27. Acceso escalar: `.at` y `.iat`

In [ ]:
print(".at:", ventas.at[0, "precio"])
print(".iat:", ventas.iat[0, ventas.columns.get_loc("precio")])

- `.at`: acceso escalar por etiqueta.
- `.iat`: acceso escalar por posición.

Son útiles cuando realmente necesitamos un único valor.

# 28. Benchmark escalar

In [ ]:
fila = 100_000
pos_precio = ventas.columns.get_loc("precio")

def acceso_loc():
    return ventas.loc[fila, "precio"]

def acceso_at():
    return ventas.at[fila, "precio"]

def acceso_iloc():
    return ventas.iloc[fila, pos_precio]

def acceso_iat():
    return ventas.iat[fila, pos_precio]

metodos = {
    "loc": acceso_loc,
    "at": acceso_at,
    "iloc": acceso_iloc,
    "iat": acceso_iat
}

resultados = []

for nombre, funcion in metodos.items():
    tiempos = medir(funcion, repeticiones=200)
    resultados.append({
        "metodo": nombre,
        "media_us": tiempos.mean() * 1_000_000
    })

benchmark_escalar = pd.DataFrame(resultados)
benchmark_escalar

## 28.1 Visualizar acceso escalar

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    benchmark_escalar["metodo"],
    benchmark_escalar["media_us"]
)
ax.set(
    title="Acceso a un único valor",
    xlabel="Método",
    ylabel="Tiempo medio (µs)"
)
plt.show()

### Interpretación

`.at` y `.iat` comunican explícitamente que se desea acceso escalar. Las diferencias absolutas suelen ser pequeñas, por lo que no deben convertirse en microoptimizaciones prematuras.

# 29. Ordenar antes de seleccionar top-N

In [ ]:
top_ventas = (
    ventas.loc[
        :,
        ["venta_id", "canal", "importe"]
    ]
    .sort_values(
        "importe",
        ascending=False
    )
    .head(10)
)

top_ventas

## 29.1 Visualizar las operaciones de mayor importe

In [ ]:
top_plot = top_ventas.sort_values("importe")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    top_plot["venta_id"].astype(str),
    top_plot["importe"]
)
ax.set(
    title="Top 10 operaciones por importe",
    xlabel="Importe",
    ylabel="venta_id"
)
plt.show()

### Interpretación

La selección de columnas se realiza antes de construir el objeto final que utilizaremos para comunicar el resultado.

# 30. Filtrar fechas correctamente

In [ ]:
inicio = pd.Timestamp("2025-02-01")
fin = pd.Timestamp("2025-02-28 23:59:59")

ventas_febrero = ventas.loc[
    ventas["fecha"].between(inicio, fin)
]

ventas_febrero[["fecha", "importe"]].head()

## 30.1 Validar el rango

In [ ]:
ventas_febrero["fecha"].agg(["min", "max"])

# 31. Selección de tipos con `select_dtypes`

In [ ]:
numericas = ventas.select_dtypes(include="number")
categoricas = ventas.select_dtypes(include=["object", "category"])

print("Numéricas:", list(numericas.columns))
print("Texto/categoría:", list(categoricas.columns))

Seleccionar columnas por dtype resulta útil para construir pipelines genéricos, aunque siempre debemos verificar que la semántica de las variables coincida con su dtype.

# 32. Un entero no siempre es una variable cuantitativa

In [ ]:
print("cliente_id dtype:", ventas["cliente_id"].dtype)
print("Media de cliente_id:", ventas["cliente_id"].mean())

La media de un identificador es computable, pero normalmente carece de interpretación.

**dtype técnico ≠ significado estadístico.**

Este principio será importante en análisis y feature engineering.

# 33. Función de auditoría de DataFrame

In [ ]:
def auditar_dataframe(datos: pd.DataFrame) -> pd.DataFrame:
    memoria = datos.memory_usage(deep=True)

    auditoria = pd.DataFrame({
        "dtype": datos.dtypes.astype(str),
        "nulos": datos.isna().sum(),
        "n_unicos": datos.nunique(dropna=False),
        "memoria_MB": memoria.drop("Index", errors="ignore") / 1024**2
    })

    auditoria["pct_nulos"] = (
        auditoria["nulos"] / len(datos) * 100
    )

    auditoria["pct_unicos"] = (
        auditoria["n_unicos"] / len(datos) * 100
    )

    return auditoria.sort_values(
        "memoria_MB",
        ascending=False
    )

auditoria = auditar_dataframe(ventas)
auditoria

## 33.1 Detectar candidatas a categoría

In [ ]:
candidatas_categoria = auditoria.loc[
    (auditoria["dtype"] == "object")
    & (auditoria["pct_unicos"] < 5)
]

candidatas_categoria

### Interpretación

La auditoría convierte una recomendación general —“mira los tipos”— en un diagnóstico reproducible y cuantificable.

# 34. Mini-práctica: optimizar y validar

In [ ]:
ventas_practica = ventas.copy()

memoria_inicial = (
    ventas_practica.memory_usage(deep=True).sum()
    / 1024**2
)

for columna in ["canal", "region"]:
    ventas_practica[columna] = (
        ventas_practica[columna]
        .astype("category")
    )

for columna in ["venta_id", "cliente_id", "unidades"]:
    ventas_practica[columna] = pd.to_numeric(
        ventas_practica[columna],
        downcast="unsigned"
    )

memoria_final = (
    ventas_practica.memory_usage(deep=True).sum()
    / 1024**2
)

print("Memoria inicial:", round(memoria_inicial, 2), "MB")
print("Memoria final:", round(memoria_final, 2), "MB")
print(
    "Reducción:",
    round((1 - memoria_final / memoria_inicial) * 100, 2),
    "%"
)

## 34.1 Validación funcional

In [ ]:
validaciones = {
    "mismas_filas": len(ventas) == len(ventas_practica),
    "mismos_importes": np.allclose(
        ventas["importe"],
        ventas_practica["importe"]
    ),
    "mismos_canales": (
        ventas["canal"].astype(str).to_numpy()
        ==
        ventas_practica["canal"].astype(str).to_numpy()
    ).all()
}

validaciones

## 34.2 Comparación gráfica final

In [ ]:
comparacion_final = pd.DataFrame({
    "version": ["Original", "Optimizada"],
    "MB": [memoria_inicial, memoria_final]
})

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    comparacion_final["version"],
    comparacion_final["MB"]
)
ax.set(
    title="Resultado de la optimización",
    ylabel="Memoria (MB)"
)
plt.show()

### Interpretación final de la práctica

La optimización se acepta porque cumple dos condiciones:

1. reduce recursos;
2. conserva los resultados relevantes.

Este doble criterio será recurrente en los siguientes laboratorios de rendimiento.

# 35. Checklist de selección eficiente

- [ ] ¿He inspeccionado `shape`, índice y dtypes?
- [ ] ¿He medido memoria con `deep=True`?
- [ ] ¿Hay strings repetitivos candidatos a `category`?
- [ ] ¿El rango permite downcasting?
- [ ] ¿He validado la precisión después de optimizar?
- [ ] ¿Uso `.loc` cuando selecciono por etiqueta?
- [ ] ¿Uso `.iloc` cuando selecciono por posición?
- [ ] ¿Evito chained indexing?
- [ ] ¿Uso `.copy()` cuando quiero independencia explícita?
- [ ] ¿Las máscaras complejas tienen nombres legibles?
- [ ] ¿Uso `isin()` para pertenencia?
- [ ] ¿Selecciono solo columnas necesarias?
- [ ] ¿Mido antes de afirmar que algo es más rápido?
- [ ] ¿Valido equivalencia entre alternativas?

# 36. Ejercicios finales


## Ejercicio 1 — Series y DataFrames

Demuestra cinco selecciones sobre ventas. Documenta el tipo, shape y significado de cada resultado. Incluye Series, DataFrame de una columna, varias columnas, fila y escalar.


In [ ]:
# TODO: crea cinco selecciones y una tabla con tipo y shape.


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


## Ejercicio 2 — Auditoría

Construye una tabla por columna con dtype, nulos, cardinalidad y memoria profunda. Define qué ocurre con una tabla vacía y cómo contabilizar el índice.


In [ ]:
def auditoria_practica(datos: pd.DataFrame) -> pd.DataFrame:
    # TODO: construye la auditoría, evitando dividir por cero.
    pass


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


## Ejercicio 3 — Categorías

Detecta automáticamente columnas object con menos del 5 % de valores únicos. Convierte en una copia y valida que valores, nulos e índice se mantienen. No conviertas identificadores solo por su dtype.


In [ ]:
def categorizar_practica(datos: pd.DataFrame, limite: float = 0.05):
    # TODO: detecta object de baja cardinalidad y devuelve copia y candidatas.
    pass
# TODO: compara valores, nulos e índice antes y después.


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


## Ejercicio 4 — Memoria

Mide la memoria antes y después de categorías y representa ambos valores. Cuantifica el ahorro y distingue memoria del objeto de memoria máxima del proceso.


In [ ]:
# TODO: mide con memory_usage(deep=True).sum() y convierte a MiB.
# TODO: grafica y calcula el porcentaje de reducción.


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


## Ejercicio 5 — Downcasting

Inspecciona rango y dtype de venta_id, cliente_id y unidades. Optimiza estos enteros y comprueba igualdad exacta de valores. Explica por qué no debe convertirse un identificador a float.


In [ ]:
# TODO: inspecciona min y max de tres columnas enteras.
# TODO: aplica downcast unsigned solo si no hay negativos.
# TODO: comprueba igualdad exacta de valores e índice.


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


## Ejercicio 6 — .loc vs .iloc

Crea un DataFrame con índice [10, 20, 30, 40]. Muestra acceso por etiqueta y posición, slices inclusivo/exclusivo y el error de una etiqueta ausente.


In [ ]:
tabla_indices_practica = pd.DataFrame({"valor": [100, 200, 300, 400]}, index=[10, 20, 30, 40])
# TODO: compara loc[20] con iloc[1]; loc[10:30] con iloc[0:3].
# TODO: captura el KeyError de loc[1].


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


## Ejercicio 7 — Error booleano

Provoca el error de usar and entre Series y corrige con &. Usa paréntesis en cada comparación y valida las filas resultantes.


In [ ]:
# TODO: intenta combinar canal == web and importe >= 1000; captura ValueError.
# TODO: corrige con &, selecciona y comprueba los resultados.


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


## Ejercicio 8 — Chained indexing

Construye una asignación encadenada y compárala con una asignación explícita usando loc. Comprueba las filas que cambian y las que deben mantenerse. No presupongas que aparezca una advertencia en todas las versiones de Pandas.


In [ ]:
demo_asignacion = pd.DataFrame({"canal": ["web", "tienda", "web"], "marca": ["sin", "sin", "sin"]})
# TODO: muestra demo[mascara]["marca"] = "revisar" y reescribe con loc.


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


## Ejercicio 9 — Máscaras con nombre

Selecciona ventas web, de regiones Norte/Este/Centro, importe >= 1000 y descuento <= 0.10. Usa cuatro máscaras descriptivas y selecciona únicamente venta_id, canal, region e importe.


In [ ]:
# TODO: define es_web, es_region_objetivo, es_importe_alto y es_descuento_bajo.
# TODO: combina las máscaras y selecciona solo las columnas necesarias.


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


## Ejercicio 10 — isin y between

Combina regiones Norte/Este y un importe entre 500 y 1000, con ambos extremos incluidos. Comprueba equivalencia con comparaciones explícitas.


In [ ]:
# TODO: combina isin y between(inclusive="both").
# TODO: compara con la forma equivalente >= y <=.


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


## Ejercicio 11 — Benchmark

Compara máscara booleana y query sobre las mismas filas y columnas. Valida equivalencia primero. Mide diez repeticiones por método, alterna el orden y grafica media y desviación estándar.


In [ ]:
# TODO: define comparar_filtros_practica y graficar_benchmark_practica.
# TODO: valida igualdad antes de medir; alterna orden y usa 10 repeticiones.


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


## Ejercicio 12 — Reto final

Utiliza las 200.000 filas de ventas: audita, detecta dtypes, mide memoria, convierte categorías y enteros, valida equivalencia, filtra, demuestra una mala asignación, corrige con loc y compara filtros. Interpreta ambos gráficos.


In [ ]:
# TODO: parte de una copia de ventas y comprueba >= 100000 filas.
# TODO: ejecuta auditoría, categorías y downcasting conservando valores.
# TODO: selecciona columnas y filtros de negocio.
# TODO: demuestra una mala asignación y corrige con loc sobre una copia.
# TODO: grafica memoria y benchmark; interpreta ambos.


### Tu respuesta / interpretación

_Escribe aquí el resultado, las comprobaciones y la interpretación._


# 37. Preguntas de reflexión


1. ¿Una `Series` y un `DataFrame` de una columna son equivalentes?
2. ¿Por qué `object` puede consumir mucha memoria?
3. ¿Cuándo tiene sentido `category`?
4. ¿Qué riesgo tiene el downcasting?
5. ¿Qué diferencia fundamental hay entre `.loc` e `.iloc`?
6. ¿Por qué no usamos `and` entre Series booleanas?
7. ¿Qué problema intenta evitar `.loc` en las asignaciones?
8. ¿Por qué seleccionar menos columnas puede ser beneficioso?
9. ¿Una alternativa más legible es necesariamente más lenta?
10. ¿Por qué debemos validar equivalencia después de optimizar?

## Tus respuestas

1. …
2. …
3. …
4. …
5. …
6. …
7. …
8. …
9. …
10. …
